# Tydzień 9: Przetwarzanie języka naturalnego (NLP) – chmury słów i modelowanie tematów 📰

| | |
|---|---|
| **Rozdział książki** | 6. *Natural Language Processing* |
| **Czas zajęć** | 90 minut |
| **Dane** | `data/bbc-text.csv` – 2225 artykułów BBC News z 5 kategorii (plik jest już w repozytorium) |
| **Biblioteki** | pandas, WordCloud, NLTK, scikit-learn (LDA), PyCaret (klasyfikacja) |

## 🎯 Cel zajęć
Po dzisiejszych zajęciach będziesz:
1. wiedzieć, czym jest **NLP** i jak zamienić tekst na liczby (**worek słów**, *bag of words*),
2. znać podstawowe kroki przygotowania tekstu: **tokenizacja**, **usuwanie stopwords**, **lematyzacja**,
3. umieć narysować **chmurę słów** (*word cloud*) i wykres najczęstszych słów,
4. rozumieć ideę **modelowania tematów** (*topic modelling*) metodą **LDA** i umieć wykryć tematy w artykułach,
5. (bonus) użyć tematów jako cech do **klasyfikacji** artykułów w PyCaret.

## ⚠️ Ważna uwaga o wersji PyCaret
W książce rozdział 6 korzysta z modułu `pycaret.nlp`. **W PyCaret 3 ten moduł został usunięty** (autor książki też o tym pisze).
Dlatego modelowanie tematów zrobimy **bezpośrednio w scikit-learn** – to ten sam algorytm LDA, tylko kilka linijek więcej.
Klasyfikację na końcu zrobimy już w PyCaret, tak jak w książce.

## 🗓️ Plan zajęć
| Czas | Co robimy |
|---|---|
| 0–20 min | 📚 Teoria: NLP, przygotowanie tekstu, LDA |
| 20–70 min | 👣 Krok po kroku |
| 70–85 min | 🧑‍💻 Ćwiczenie samodzielne – ✅ **obowiązkowe** (zaliczenie) |
| 85–90 min | 📝 Podsumowanie |
| poza zajęciami | 🏋️ Część 4: ćwiczenia dodatkowe (trudniejsze, nieobowiązkowe) – dla szybszych osób lub do domu |

---
# 📚 Część 1: Wprowadzenie teoretyczne

## Co to jest NLP? 💬
**Przetwarzanie języka naturalnego** (*Natural Language Processing, NLP*) to dziedzina na styku językoznawstwa i ML.
Uczy komputery „rozumieć” język, którym mówią ludzie. Przykłady: tłumacz Google, filtr spamu, asystenci głosowi,
analiza opinii klientów, czatboty.

## Słownik pojęć NLP 📚
| Pojęcie | Znaczenie | Przykład |
|---|---|---|
| **korpus** (*corpus*) | cały zbiór tekstów | 2225 artykułów BBC |
| **dokument** (*document*) | jeden tekst | jeden artykuł |
| **token** | najmniejsza jednostka tekstu (zwykle słowo) | „phones” |
| **tokenizacja** (*tokenization*) | dzielenie tekstu na tokeny | „I like cats” → [I, like, cats] |
| **stopwords** | bardzo częste słowa bez treści | the, a, is, and, of |
| **lematyzacja** (*lemmatization*) | sprowadzanie słowa do formy słownikowej (lematu) | phones → phone, mice → mouse |
| **n-gram** | ciąg *n* kolejnych słów | bigram: „mobile phone” |

## Jak zamienić tekst na liczby? – worek słów 🛍️
Modele ML rozumieją tylko liczby. Najprostszy sposób to **worek słów** (*bag of words*): wyobraź sobie, że wycinasz
z artykułu wszystkie słowa nożyczkami, wrzucasz do worka i **liczysz**, ile razy wystąpiło każde słowo. Kolejność słów
ginie, ale zostaje informacja, **o czym** jest tekst. Wynik to tabela: wiersze = dokumenty, kolumny = słowa, komórki = liczba wystąpień.

## Modelowanie tematów i LDA 🍹
**Modelowanie tematów** (*topic modelling*) to uczenie **nienadzorowane**: model sam odkrywa ukryte **tematy** w zbiorze tekstów.

**LDA** (*Latent Dirichlet Allocation*, ukryta alokacja Dirichleta) zakłada, że:
- każdy **temat** to „przepis” – lista słów z wagami (np. temat „sport”: *game, team, win, match…*),
- każdy **dokument** to **koktajl** 🍹 z kilku tematów w różnych proporcjach (np. 70% „sport” + 30% „biznes” – artykuł o transferze piłkarza za miliony).

LDA na podstawie samych słów zgaduje i przepisy tematów, i proporcje tematów w każdym artykule.
Liczbę tematów (np. 5) musimy podać sami – podobnie jak K w K-Means.

(Uwaga: to **inne LDA** niż *Linear Discriminant Analysis* z tygodnia 6 – ten sam skrót, zupełnie inny model!)

## Plan dzisiejszej „taśmy produkcyjnej” 🏭
```
artykuły → przygotowanie tekstu → worek słów → LDA (5 tematów) → proporcje tematów → klasyfikacja kategorii (PyCaret)
```

---
# 👣 Część 2: Krok po kroku

### Krok 1: Importy

In [ ]:
# Wyłączamy ostrzeżenia (warnings), żeby nie zaśmiecały ekranu.
# Ostrzeżenie to NIE błąd – to tylko podpowiedź dla programistów bibliotek.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Tabele i wykresy
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# WordCloud – chmury słów; STOPWORDS – gotowa lista angielskich stopwords z tej biblioteki
from wordcloud import WordCloud, STOPWORDS

### Krok 2: Wczytujemy artykuły BBC 📰
Plik `bbc-text.csv` znajduje się w folderze `data/` repozytorium (pochodzi z repozytorium książki; autorzy zbioru:
D. Greene i P. Cunningham, University College Dublin).

In [ ]:
# read_csv wczytuje plik CSV do tabeli; '../data/' = folder data piętro wyżej niż folder notebooks
data = pd.read_csv('../data/bbc-text.csv')
data.head()

In [ ]:
# Dwie kolumny: kategoria i pełny tekst artykułu; brak braków danych
data.info()

In [ ]:
# Zobaczmy początek pierwszego artykułu (500 pierwszych znaków) – teksty są już zapisane małymi literami
print(data.loc[0, 'text'][:500])

### Krok 3: EDA – kategorie i długość artykułów 📊

In [ ]:
# Liczba artykułów w każdej kategorii (business, entertainment, politics, sport, tech)
data['category'].value_counts().plot(kind='bar', color=['C0', 'C1', 'C2', 'C3', 'C4'])
plt.show()

🔎 Klasy **nie są idealnie zbalansowane**: najwięcej jest artykułów sportowych i biznesowych (ok. 510), najmniej
rozrywkowych (ok. 390). Zajmiemy się tym przy klasyfikacji (`fix_imbalance=True`).

In [ ]:
# Liczba słów w każdym artykule: split() dzieli tekst po spacjach, len liczy elementy
n_words = data['text'].str.split().str.len()
n_words.describe()

In [ ]:
# Histogram długości artykułów
n_words.hist(bins=50)
plt.xlabel('liczba słów w artykule')
plt.show()

### Krok 4: Chmura słów ☁️
W **chmurze słów** wielkość słowa zależy od tego, jak często występuje. Narysujmy chmurę dla kategorii **tech**.

In [ ]:
# Własna lista stopwords: gotowa lista + słowa bardzo częste w gazetach, które nic nie mówią o temacie
my_stopwords = STOPWORDS | {'said', 'will', 'one', 'also', 'says', 'year', 'new', 'many', 'now', 'two'}

In [ ]:
# Ustawienia chmury: rozmiar obrazka, stopwords do pominięcia, białe tło, słowa min. 3-literowe, max 100 słów
wc = WordCloud(width=1200, height=800, stopwords=my_stopwords, background_color='white',
               min_word_length=3, max_words=100)

In [ ]:
# Sklejamy wszystkie artykuły z kategorii tech w jeden długi tekst (' '.join łączy teksty spacjami)
text_tech = ' '.join(data.query("category == 'tech'")['text'])

In [ ]:
# Generujemy i wyświetlamy chmurę; axis('off') ukrywa osie
plt.figure(figsize=(12, 8))
plt.imshow(wc.generate(text_tech))
plt.axis('off')
plt.show()

🔎 Dominują słowa związane z technologią: *people, game, technology, mobile, phone, software, users…*
💡 Spróbuj usunąć własne stopwords (użyj `stopwords=STOPWORDS`) – wtedy chmurę zdominują słowa *said*, *will* i *one*,
bo w artykułach prasowych ciągle kogoś się cytuje. Dobór stopwords to ważna decyzja w NLP!

In [ ]:
# Najczęstsze słowa w CAŁYM korpusie: process_text zwraca słownik {słowo: liczba wystąpień}
freq = wc.process_text(' '.join(data['text']))
top20 = pd.Series(freq).sort_values().tail(20)

In [ ]:
# Wykres słupkowy 20 najczęstszych słów
top20.plot(kind='barh', figsize=(8, 6))
plt.show()

### Krok 5: Przygotowanie tekstu krok po kroku (na jednym zdaniu) 🧹
Zobaczmy, co dzieje się z tekstem przed modelowaniem. Użyjemy biblioteki **NLTK** (*Natural Language Toolkit*).

In [ ]:
# NLTK i moduł re (wyrażenia regularne – "wzorce" do wyszukiwania tekstu)
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
# Pobieramy słowniki NLTK (tylko za pierwszym razem; quiet=True – bez komunikatów)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

In [ ]:
# Przykładowe zdanie
sentence = "The new Mobile Phones were selling better than expected in 2004, analysts said!"

In [ ]:
# 1) Małe litery + usunięcie cyfr i znaków specjalnych: [^a-z ] = "wszystko, co NIE jest literą a–z ani spacją"
clean = re.sub(r'[^a-z ]', ' ', sentence.lower())
clean

In [ ]:
# 2) Tokenizacja – dzielimy tekst na słowa (tokeny)
tokens = clean.split()
tokens

In [ ]:
# 3) Usuwanie stopwords – wyrzucamy słowa z angielskiej listy NLTK (the, were, than, in…)
stop = set(stopwords.words('english'))
tokens = [t for t in tokens if t not in stop]
tokens

In [ ]:
# 4) Lematyzacja – forma słownikowa (phones -> phone, analysts -> analyst)
lemmatizer = WordNetLemmatizer()
[lemmatizer.lemmatize(t) for t in tokens]

💡 PyCaret 2 (w książce) robił te kroki automatycznie w `setup()`. My w kolejnym kroku zlecimy większość z nich
klasie `CountVectorizer` ze scikit-learn.

### Krok 6: Worek słów (*bag of words*) 🛍️
Najpierw zobaczmy, jak to działa na dwóch krótkich „dokumentach”:

In [ ]:
# CountVectorizer zamienia teksty na tabelę: wiersze = dokumenty, kolumny = słowa, wartości = liczba wystąpień
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# Mini-przykład: dwa krótkie dokumenty
toy = ['the cat eats fish', 'the dog eats a bone and the cat sleeps']
toy_vec = CountVectorizer()

In [ ]:
# fit_transform "uczy się" słownika i liczy słowa; toarray() zamienia wynik na zwykłą tabelę liczb
toy_counts = toy_vec.fit_transform(toy)
pd.DataFrame(toy_counts.toarray(), columns=toy_vec.get_feature_names_out())

🔎 Drugi dokument ma `the = 2`, bo słowo „the” wystąpiło w nim dwa razy. Słowo „a” zniknęło – domyślnie pomijane są słowa jednoliterowe.

Teraz to samo dla wszystkich 2225 artykułów, z dodatkowymi ustawieniami:

In [ ]:
vectorizer = CountVectorizer(
    stop_words='english',                  # usuń angielskie stopwords
    token_pattern=r'\b[a-z]{3,}\b',        # tokeny = tylko słowa z liter, min. 3 litery (bez liczb typu "2004")
    max_df=0.5,                            # pomiń słowa obecne w ponad 50% artykułów (zbyt ogólne, np. "said")
    min_df=5,                              # pomiń słowa obecne w mniej niż 5 artykułach (zbyt rzadkie)
    max_features=5000,                     # zostaw 5000 najczęstszych słów
)

In [ ]:
# dtm = document-term matrix (macierz dokument–słowo)
dtm = vectorizer.fit_transform(data['text'])
dtm.shape

🔎 Mamy tabelę **2225 artykułów × 5000 słów**. Większość komórek to zera (artykuł używa tylko małej części słownika) –
dlatego scikit-learn przechowuje ją w oszczędnym formacie „rzadkim” (*sparse matrix*).

### Krok 7: Modelowanie tematów – LDA 🍹

In [ ]:
# LatentDirichletAllocation = model LDA; n_components=5 – szukamy 5 tematów (tyle, ile kategorii)
from sklearn.decomposition import LatentDirichletAllocation
lda = LatentDirichletAllocation(n_components=5, random_state=6842)

In [ ]:
# Trenujemy LDA (ok. 5–15 sekund) i od razu dostajemy proporcje tematów w każdym artykule
topic_proportions = lda.fit_transform(dtm)

In [ ]:
# Lista słów (kolumn worka słów) – potrzebna, żeby odczytać, co oznacza każdy temat
words = vectorizer.get_feature_names_out()

In [ ]:
# 10 słów o największej wadze w każdym temacie: argsort() sortuje wagi, [-10:][::-1] bierze 10 największych
for i, weights in enumerate(lda.components_):
    top_words = words[weights.argsort()[-10:][::-1]]
    print(f"Temat {i}:", ', '.join(top_words))

🔎 **Nazwij tematy!** Model nie zna nazw – dał nam tylko listy słów. Spróbuj odgadnąć, który temat to polityka
(*government, labour, election, blair…*), sport (*game, england, team, win…*), gospodarka (*company, market, bank, shares…*),
technologia (*music, technology, mobile, users…*), a który – rozrywka (*film, awards, director, star…*).

In [ ]:
# Tabela proporcji tematów: każdy wiersz sumuje się do 1 (100%)
cols = ['Topic_0', 'Topic_1', 'Topic_2', 'Topic_3', 'Topic_4']
topics = pd.DataFrame(topic_proportions, columns=cols).round(3)
topics['category'] = data['category']
topics.head(10)

In [ ]:
# Dominujący temat = temat z największą proporcją; idxmax(axis=1) zwraca nazwę kolumny z maksimum w wierszu
dominant = topics[cols].idxmax(axis=1)

In [ ]:
# Tabela krzyżowa: prawdziwa kategoria (wiersze) vs dominujący temat (kolumny)
pd.crosstab(topics['category'], dominant)

🔎 Mimo że LDA **nie znał kategorii**, każda kategoria ma swój wyraźnie dominujący temat – szczególnie polityka i sport!
Najwięcej „przecieków” jest między rozrywką a technologią (muzyka, gry i telewizja cyfrowa pasują do obu)
oraz z biznesu do innych tematów (firmy działają przecież w każdej branży).

### Krok 8 (bonus): Klasyfikacja artykułów na podstawie tematów – PyCaret 🤖
Proporcje 5 tematów to teraz **5 liczbowych cech** każdego artykułu. Czy na ich podstawie model przewidzi kategorię?
To powtórka z tygodni 5–6.

In [ ]:
# Moduł klasyfikacji PyCaret
from pycaret.classification import *

In [ ]:
# fix_imbalance=True – wyrównuje liczebność klas w zbiorze treningowym (metoda SMOTE tworzy sztuczne przykłady mniejszych klas)
classf = setup(data=topics, target='category', fix_imbalance=True, train_size=0.8, session_id=3100)

In [ ]:
# Model CatBoost – jak w książce (⏳ ok. 30–60 s)
catboost = create_model('catboost')

In [ ]:
# Sprawdzian na zbiorze testowym
predictions = predict_model(catboost)

In [ ]:
# Macierz pomyłek – które kategorie są mylone? (0=business, 1=entertainment, 2=politics, 3=sport, 4=tech)
plot_model(catboost, plot='confusion_matrix')

🔎 Model osiąga dokładność ok. **90%**, choć zna tylko 5 liczb na artykuł! Najczęściej bierze rozrywkę (1) za technologię (4) –
tak jak przewidywaliśmy na podstawie tabeli krzyżowej. ⚠️ Jeśli przy niektórych metrykach widzisz **AUC = 0**, to znany błąd PyCaret 3.3 przy wielu klasach (patrz tydzień 6).

---
# 🧑‍💻 Część 3: Ćwiczenie dla Ciebie
> ✅ **Ćwiczenie obowiązkowe** – na jego podstawie zaliczasz dzisiejsze zajęcia (zaliczone / niezaliczone).

**Zadanie:** Poeksperymentuj z tekstem.

- **a)** Narysuj chmurę słów dla kategorii **sport** (użyj obiektu `wc` z kroku 4).
- **b)** Wytrenuj nowy model LDA z **3 tematami** (`n_components=3`, `random_state=6842`) na tej samej macierzy `dtm`
  i wypisz 10 najważniejszych słów każdego tematu.
- **c)** Narysuj tabelę krzyżową kategorii i dominującego tematu. Które kategorie „skleiły się” w jeden temat?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- a) Skopiuj komórki z kroku 4 i zmień `'tech'` na `'sport'`.
- b) Skopiuj komórki z kroku 7: `LatentDirichletAllocation(n_components=3, ...)`, potem `fit_transform(dtm)` i pętlę `for` z top-10 słowami.
- c) `pd.DataFrame(...).idxmax(axis=1)` i `pd.crosstab(data['category'], ...)`.

</details>

In [ ]:
# ✍️ Twój kod tutaj (możesz dodawać kolejne komórki przyciskiem "+ Code")

### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

🌟 **Dla chętnych:** chcesz sklasyfikować **własny** tekst? Zobacz ćwiczenie 3 w części 4 poniżej.

---
# 🏋️ Część 4: Ćwiczenia dodatkowe – trochę trudniejsze
Dla osób, które skończyły ćwiczenie podstawowe, albo **do samodzielnej pracy w domu**.
Ćwiczenia z tej części są **nieobowiązkowe** – nie są wymagane do zaliczenia zajęć.

Te zadania mają mniej podpowiedzi w treści, łączą kilka poznanych narzędzi i czasem wprowadzają **jedną nową funkcję**
(zawsze opisaną w podpowiedzi). Poziomy trudności: ⭐⭐ – średnie, ⭐⭐⭐ – wyzwanie.

## ⭐⭐ Ćwiczenie 2: Najczęstsze bigramy
Pojedyncze słowa gubią sens wyrażeń: „prime” i „minister” osobno znaczą co innego niż razem. **Bigram** to para kolejnych słów.

- **a)** Utwórz worek **bigramów** dla wszystkich artykułów (stopwords angielskie, tylko słowa z liter min. 3-literowe,
  bigram musi wystąpić w co najmniej 5 artykułach).
- **b)** Znajdź **15 najczęstszych** bigramów w całym korpusie i narysuj je na wykresie słupkowym.
- **c)** Jakich tematów dotyczą? Czy widać „ślady” epoki, z której pochodzą artykuły (2004–2005)?

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- `CountVectorizer(ngram_range=(2, 2), ...)` – tokenami są pary słów zamiast pojedynczych słów.
  Pozostałe parametry skopiuj z kroku 6 (`stop_words`, `token_pattern`, `min_df=5`).
- Suma wystąpień każdego bigramu w całym korpusie: `macierz.sum(axis=0).A1` (`.A1` zamienia wynik na zwykłą listę liczb).
- Połącz liczby z nazwami: `pd.Series(liczby, index=wektoryzator.get_feature_names_out())`, potem `.sort_values().tail(15)`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

## ⭐⭐⭐ Ćwiczenie 3: Własny klasyfikator nagłówków – cały potok od tekstu do kategorii
Mamy już wszystkie elementy: `vectorizer` (worek słów), `lda` (tematy) i `catboost` (klasyfikator na proporcjach tematów).
Złóż z nich **potok** (*pipeline*), który dla **nowego** tekstu przewidzi kategorię.

- **a)** Napisz 5 krótkich „artykułów” po angielsku – po jednym z każdej kategorii (politics, sport, business, tech, entertainment).
- **b)** Przepuść je przez potok: tekst → worek słów → proporcje tematów → tabela z kolumnami `Topic_0` … `Topic_4`.
- **c)** Przewidź kategorie modelem `catboost`. Czy wszystkie są poprawne? Spróbuj też „podchwytliwego” tekstu z dwóch dziedzin naraz.

<details>
<summary><b>💡 Podpowiedź</b> (kliknij, aby rozwinąć)</summary>

- Na nowych danych używamy `transform`, **nie** `fit_transform` – model już się nauczył, teraz tylko go stosujemy:
  `counts = vectorizer.transform(lista_tekstow)`, potem `lda.transform(counts)`.
- Tabela z proporcjami tematów: `pd.DataFrame(proporcje, columns=cols)` (lista `cols` jest z kroku 7).
- Predykcja: `predict_model(catboost, data=tabela)` – kolumny `prediction_label` i `prediction_score`.

</details>

In [ ]:
# ✍️ Twój kod tutaj

#### ✍️ Twoje odpowiedzi
*Kliknij dwukrotnie tę komórkę i wpisz odpowiedzi na pytania z zadania oraz swoje wnioski – własnymi słowami.*

-

---
# 📝 Podsumowanie
- **NLP** zamienia tekst na liczby; najprostszy sposób to **worek słów** (`CountVectorizer`).
- Przygotowanie tekstu: małe litery, usunięcie cyfr i znaków, **tokenizacja**, usunięcie **stopwords**, **lematyzacja**.
- **Chmura słów** i wykres częstości to szybka EDA dla tekstu.
- **LDA** (*Latent Dirichlet Allocation*) odkrywa tematy bez etykiet: temat = lista słów, dokument = mieszanka tematów.
- Proporcje tematów mogą być **cechami** dla klasyfikatora – tak łączymy uczenie nienadzorowane z nadzorowanym.
- Moduł `pycaret.nlp` istniał tylko w PyCaret 2 – w PyCaret 3 tematy liczymy w scikit-learn.

## 📖 Słowniczek PL / EN
| Polski | English | Znaczenie |
|---|---|---|
| przetwarzanie języka naturalnego | Natural Language Processing (NLP) | analiza tekstu przez komputer |
| korpus | corpus | zbiór dokumentów |
| tokenizacja | tokenization | dzielenie tekstu na słowa |
| słowa nieistotne | stopwords | bardzo częste słowa bez treści |
| lematyzacja | lemmatization | sprowadzanie słów do formy podstawowej |
| worek słów | bag of words | zliczanie słów bez kolejności |
| macierz dokument–słowo | document-term matrix | tabela: artykuły × słowa |
| modelowanie tematów | topic modelling | odkrywanie tematów w tekstach |
| ukryta alokacja Dirichleta | Latent Dirichlet Allocation (LDA) | model tematów: dokument = mieszanka tematów |
| chmura słów | word cloud | wizualizacja częstości słów |
| niezbalansowane klasy | imbalanced classes | klasy o różnej liczebności |

➡️ **Za tydzień:** **szeregi czasowe** – prognozujemy stężenie CO₂ w atmosferze 🌍.